In [11]:
import pycrfsuite # our last attempt does not work so we changed plans and try with geekforgeeks method
import conllu # our id data not just a plain text its complex table format where every word has 10+ columns of language info. This lib saves us from writing a confusing Regex parser to read it
from sklearn.metrics import accuracy_score

In [12]:
def load_conllu_data(file_path):
    data_list = [] # this is empty list but in the end it will hold our cleaned data

    with open(file_path, 'r', encoding='utf-8') as f: # openning a file in a readable mod with the utf-8 encoding 

        for sentence in conllu.parse_incr(f): # this is read file sentence by sentence instead of loading all at once into ram
            words = []
            tags = []
            for token in sentence: # we will loop through every token
                words.append(token['form']) # actual word seen in text
                tags.append(token['upos']) # universal part of speech tag    
            data_list.append((words, tags)) # this row func easier to show ["makan", "nasi"], ["verb", "noun"]
    return data_list

In [13]:
train_data = load_conllu_data('id_gsd-ud-train.conllu')
dev_data = load_conllu_data('id_gsd-ud-dev.conllu')
test_data = load_conllu_data('id_gsd-ud-test.conllu')

print(f"Training sentences: {len(train_data)}")
print(f"Dev sentences: {len(dev_data)}")
print(f"Test sentences: {len(test_data)}")
print(f"Sample sentence: {train_data[0]}")

Training sentences: 4477
Dev sentences: 559
Test sentences: 557
Sample sentence: (['Sembungan', 'adalah', 'sebuah', 'desa', 'yang', 'terletak', 'di', 'kecamatan', 'Kejajar', ',', 'kabupaten', 'Wonosobo', ',', 'Jawa', 'Tengah', ',', 'Indonesia', '.'], ['PROPN', 'AUX', 'DET', 'NOUN', 'PRON', 'VERB', 'ADP', 'NOUN', 'PROPN', 'PUNCT', 'NOUN', 'PROPN', 'PUNCT', 'PROPN', 'PROPN', 'PUNCT', 'PROPN', 'PUNCT'])


In [ ]:
def word_features(sent, i):
    word = sent[i] # for now its list of strings
    
    features = {
        'bias': 1.0, # A baseline constant (like 'b' in y=mx+b)
        'word.lower()': word.lower(), # it makes all of it lowercase
        'word[-3:]': word[-3:],     
        'word[-2:]': word[-2:], # there are some suffixes we need to remove
        
        'word.isupper()': word.isupper(),
        'word.istitle()': word.istitle(),
        'word.isdigit()': word.isdigit(), # this part have some checks like is it uppercase is number
    }
    
    if i > 0:
        word1 = sent[i-1]
        features.update({
            '-1:word.lower()': word1.lower(), # look before word is it 
            '-1:word.istitle()': word1.istitle(), # was the word is a name 
        })
    else:
        features['BOS'] = True # begginnig of the sentence
        
    if i < len(sent)-1: # it will check we are on the last word or not 
        word1 = sent[i+1] # than grab next 
        features.update({
            '+1:word.lower()': word1.lower(),
            '+1:word.istitle()': word1.istitle(),
        })
    else:
        features['EOS'] = True # if it last word tag with en of sentence
    return features

def sent2features(sent):
    return [word_features(sent, i) for i in range(len(sent))] # run this func for every word in sentence 

def sent2labels(sent):
    return [label for token, label in sent] # extract just the tags

print("Extracting features") # as we know we will create 4 varible  

X_train = [sent2features(words) for words, tags in train_data]
y_train = [tags for words, tags in train_data]

X_test = [sent2features(words) for words, tags in test_data]
y_test = [tags for words, tags in test_data]

print("Training CRF") 	
trainer = pycrfsuite.Trainer(verbose=False) # it will create empty crf  model 

for xseq, yseq in zip(X_train, y_train): # it will meet words and tags 
    trainer.append(xseq, yseq)

trainer.set_params({
    'c1': 0.1, # for ignoring noise 
    'c2': 0.1, # prevents overfitting
    'max_iterations':100, # it will stop after 100 iterations to save time 
    'feature.possible_transitions': True # it will allow model to predit tags it never saw on trainnig
})

trainer.train('indonesian_pos.crfsuite') # it do the math can save in your disk

tagger = pycrfsuite.Tagger() # creates predictor object
tagger.open('indonesian_pos.crfsuite') # Loads the saved model file back into memory

y_pred = [tagger.tag(xseq) for xseq in X_test] # predict tags for test	sentences

flat_true = [tag for sent in y_test for tag in sent]	
flat_pred = [tag for sent in y_pred for tag in sent]

acc = accuracy_score(flat_true, flat_pred) # compare it 
print(f"CRF Accuracy: {acc:.2%}")

Extracting features
Training CRF
CRF Accuracy: 92.28%
